# 04 — Feature Engineering

For each detected event, extract numerical features that describe the electrical behavior of the load that switched.

**Features we extract:**
- `delta_power`: how much did power change at the event?
- `steady_state_mean/std`: what is the stable power level after the switch?
- `rise_time_seconds`: how quickly did power reach steady state? (motor vs resistive)
- `power_before_mean`: what was already running before this event?
- `time_of_day_hour`: when did this happen?
- `is_nighttime`: binary — after 10pm or before 6am?
- `direction`: ON or OFF event?

**Then we assign labels** from the ground-truth appliance sub-meters.
For each event, we look at which appliance had the largest power change at that timestamp.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from utils.signal_utils import extract_event_features

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
df = pd.read_csv('../data/processed/house1_7days.csv', index_col=0, parse_dates=True)
events = pd.read_csv('../data/processed/detected_events.csv', parse_dates=['timestamp'])

print(f'Events to process: {len(events)}')
print(f'Appliance columns: {[c for c in df.columns if c != "mains"]}')

## Assign ground-truth labels

For each detected event, we need to know which appliance caused it.

Strategy: at each event timestamp, find the appliance with the largest absolute change in power.
This is a simple but effective label assignment — the appliance that changed the most is most likely the one that switched.

In [ ]:
appliance_cols = [c for c in df.columns if c != 'mains']

def assign_label(timestamp, df, appliance_cols, window=5):
    '''
    Find which appliance had the largest power change around this timestamp.
    '''
    t_start = timestamp - pd.Timedelta(seconds=window)
    t_end   = timestamp + pd.Timedelta(seconds=window)
    window_df = df[appliance_cols][t_start:t_end]

    if len(window_df) < 2:
        return 'unknown'

    changes = window_df.diff().abs().max()
    best = changes.idxmax()

    # only assign label if the change is meaningful
    if changes[best] < 20:
        return 'unknown'
    return best

# test on first event
test_ts = events['timestamp'].iloc[0]
print('Label for first event:', assign_label(test_ts, df, appliance_cols))

In [ ]:
# assign labels to all events
print('Assigning labels...')
events['label'] = events['timestamp'].apply(
    lambda ts: assign_label(ts, df, appliance_cols)
)

print('\nLabel distribution:')
print(events['label'].value_counts())

## Extract features for each event

In [ ]:
mains = df['mains']

feature_rows = []
for _, event in events.iterrows():
    feat = extract_event_features(mains, event['timestamp'],
                                   window_before=30, window_after=60)
    if feat is None:
        continue
    feat['direction'] = 1 if event['direction'] == 'ON' else 0
    feat['label'] = event['label']
    feature_rows.append(feat)

feature_df = pd.DataFrame(feature_rows)

# drop unknown labels
feature_df = feature_df[feature_df['label'] != 'unknown']

# drop appliances with very few events — not enough to train on
min_events = 10
counts = feature_df['label'].value_counts()
valid_labels = counts[counts >= min_events].index
feature_df = feature_df[feature_df['label'].isin(valid_labels)]

print(f'Feature matrix: {feature_df.shape}')
print('\nEvents per appliance:')
print(feature_df['label'].value_counts())

## Visualize: Do features separate the classes?

Before training any model, check if the features actually discriminate between appliances.
If they don't, the model won't either.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# 1. delta_power by appliance
feature_df.boxplot(column='delta_power_abs', by='label', ax=axes[0], rot=30)
axes[0].set_title('Delta Power by Appliance', fontsize=10)
axes[0].set_xlabel('')
axes[0].set_ylabel('|ΔPower| (W)')

# 2. steady state mean by appliance
feature_df.boxplot(column='steady_state_mean', by='label', ax=axes[1], rot=30)
axes[1].set_title('Steady State Power by Appliance', fontsize=10)
axes[1].set_xlabel('')
axes[1].set_ylabel('Mean Power After Event (W)')

# 3. rise time by appliance
feature_df.boxplot(column='rise_time_seconds', by='label', ax=axes[2], rot=30)
axes[2].set_title('Rise Time by Appliance', fontsize=10)
axes[2].set_xlabel('')
axes[2].set_ylabel('Rise Time (seconds)')

for ax in axes:
    ax.title.set_fontsize(10)
plt.suptitle('Feature Distributions by Appliance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/04_feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# correlation heatmap of features
numeric_features = feature_df.select_dtypes(include=np.number).drop(columns=['time_of_day_hour','is_nighttime','direction'], errors='ignore')

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(numeric_features.corr(), dtype=bool))
sns.heatmap(numeric_features.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, ax=ax, annot_kws={'size':8})
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/04_feature_correlation.png', dpi=150)
plt.show()

In [ ]:
# save feature matrix
feature_df.to_csv('../data/processed/feature_matrix.csv', index=False)
print(f'Saved feature matrix: {feature_df.shape}')